# Energy Demand Capacity Risk — Data → Decision (Phase 1)

## Objective
This project analyzes ERCOT hourly electricity load data to identify seasonal demand patterns and flag high-risk capacity hours.  
The goal is not only to explore the data, but to transform raw grid load records into **decision-ready insights** that can support future forecasting and early-warning systems.

---

## Dataset
**Source:** ERCOT Hourly Load Archives  
**Coverage:** 2021 – 2026  
**Granularity:** Hourly  
**Rows:** ~43,800 records

The dataset contains system-wide load and regional load zones (Coast, East, North, South, West, etc.).

---

## Key Data Engineering Challenges
- Mixed timestamp formats
- Utility-style `24:00` hour notation
- Daylight Saving Time (DST) duplicate hours
- Null / malformed timestamps

These were normalized to produce a **continuous hourly time index** suitable for lag and rolling window calculations.

---

## Pipeline Structure
The project follows a layered SQL modeling approach:

**Staging**
- `stg_ercot_load` → Timestamp normalization and type casting
- `stg_ercot_load_1h` → One row per hour (DST duplicates averaged)

**Intermediate**
- `int_time_features` → Year, month, weekday, hour, weekend flags
- `int_demand_features` → Lag features (1h, 24h, 168h) and rolling averages

**Mart / Decision Layer**
- `fct_capacity_risk` → Capacity risk labeling using percentile thresholds

---

## Decision Output
Each hour is categorized into:
- **LOW**
- **MEDIUM**
- **HIGH**
- **EXTREME**

Risk levels are derived from historical load percentiles (P90, P95, P99), providing a simple capacity-stress indicator.

---

## Stack
- Databricks SQL
- Layered SQL Models (staging → intermediate → mart)
- Time-series feature engineering

---

## Next Phase
- Demand forecasting model
- Peak recall evaluation
- Early-warning lead-time analysis


In [0]:
-- models/staging/stg_ercot_load.sql
-- Source: ercot.raw_ercot_hourly_load
-- Output: one row per record (may include DST duplicates; handled in _1h model)

CREATE OR REPLACE VIEW ercot.stg_ercot_load AS
WITH base AS (
  SELECT
    hour_ending,

    -- Strip trailing " DST" (e.g., "11/06/2022 02:00 DST" -> "11/06/2022 02:00")
    regexp_replace(trim(hour_ending), '\\s*DST\\s*$', '') AS hour_nodst,

    -- Flag rows that were labeled DST (for auditing)
    CASE WHEN hour_ending LIKE '%DST%' THEN 1 ELSE 0 END AS is_dst_flag,

    coast, east, fwest, north, ncent, south, scent, west, ercot
  FROM ercot.raw_ercot_hourly_load
  WHERE hour_ending IS NOT NULL
),
norm AS (
  SELECT
    is_dst_flag,

    -- Normalize utility-style "24:00" -> next day "00:00"
    CASE
      WHEN hour_nodst RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4} 24:00$' THEN
        date_format(
          date_add(to_date(substr(hour_nodst, 1, 10), 'MM/dd/yyyy'), 1),
          'MM/dd/yyyy'
        ) || ' 00:00'
      ELSE hour_nodst
    END AS hour_norm,

    coast, east, fwest, north, ncent, south, scent, west, ercot
  FROM base
),
parsed AS (
  SELECT
    is_dst_flag,

    COALESCE(
      -- format A: MM/dd/yyyy HH:mm
      try_to_timestamp(hour_norm, 'MM/dd/yyyy HH:mm'),
      -- format B: yyyy-MM-dd HH:mm:ss
      try_to_timestamp(hour_norm, 'yyyy-MM-dd HH:mm:ss'),
      -- format C: yyyy-MM-dd HH:mm
      try_to_timestamp(hour_norm, 'yyyy-MM-dd HH:mm')
    ) AS hour_ts,

    CAST(coast AS DOUBLE) AS coast_mw,
    CAST(east  AS DOUBLE) AS east_mw,
    CAST(fwest AS DOUBLE) AS fwest_mw,
    CAST(north AS DOUBLE) AS north_mw,
    CAST(ncent AS DOUBLE) AS ncent_mw,
    CAST(south AS DOUBLE) AS south_mw,
    CAST(scent AS DOUBLE) AS scent_mw,
    CAST(west  AS DOUBLE) AS west_mw,
    CAST(ercot AS DOUBLE) AS ercot_mw,

    hour_norm AS hour_raw_clean
  FROM norm
)
SELECT *
FROM parsed
WHERE hour_ts IS NOT NULL;
